# Accuracy Simulations
**Paper:** *Beyond Correlation: Learning Supervised, Sample-Distinct, and Eigenimage-Interpretable Representation*  
**arXiv:** https://arxiv.org/abs/2501.XXXXX  

This notebook reproduces Tables 3 and 8 from the paper. All models are evaluated with k-NN (k=50) in latent space.

> **Cite our paper:** Moattari, M. (2025). *Beyond Correlation...* arXiv:2501.XXXXX

In [ ]:
# ─── Setup ───
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent.parent))
!pip install -q tqdm scikit-learn pyyaml


## 1. Seed and data loading
We fix the random seed to 42 and use the standard MNIST 85/15 split.

![Data split diagram](../figures/data_split.png)


In [ ]:
import torch, numpy as np
import sys; sys.path.insert(0, "../..")
from utils.seed import set_seed
from utils.data_utils import load_mnist, make_loaders
SEED = 42
set_seed(SEED)
train_ds, test_ds = load_mnist()
train_loader, test_loader = make_loaders(train_ds, test_ds, batch_size=64, seed=SEED)
print(f"Train: {len(train_ds)} samples | Test: {len(test_ds)} samples")


## 2. Train and evaluate WDIWCD (optimal hyperparameters)
**Optimal settings:** LR=0.01, Batch=128, MaxIter=1000, HistBins=32, Components=16, PerClassWeight=0.7

![WDIWCD vs WDDWCC cluster plot](../figures/cluster_comparison.png)


In [ ]:
from models.wdiwcd import WDIWCD
from utils.metrics import knn_accuracy, entropy_score
from tqdm import tqdm

set_seed(SEED)
input_dim = train_ds[0][0].shape[0]   # 784
model_wdiwcd = WDIWCD(input_dim, n_components=16, hist_bins=32, n_classes=10)
opt = torch.optim.Adam(model_wdiwcd.parameters(), lr=0.01)

for epoch in tqdm(range(100), desc="WDIWCD"):   # use 1000 for full reproduction
    for x, y in train_loader:
        opt.zero_grad()
        loss = model_wdiwcd.compute_loss(x, y, perclass_weight=0.7)
        if not torch.isnan(loss): loss.backward(); opt.step()

model_wdiwcd.eval()
tr_x = torch.stack([train_ds[i][0] for i in range(len(train_ds))])
tr_y = np.array([train_ds[i][1] for i in range(len(train_ds))])
te_x = torch.stack([test_ds[i][0] for i in range(len(test_ds))])
te_y = np.array([test_ds[i][1] for i in range(len(test_ds))])
z_tr = model_wdiwcd.transform(tr_x)
z_te = model_wdiwcd.transform(te_x)
acc  = knn_accuracy(z_tr, tr_y, z_te, te_y, k=50)
ent  = entropy_score(z_te)
print(f"WDIWCD  →  kNN Accuracy: {acc*100:.1f}%  |  Entropy Score: {ent:.4f}")


## 3. Train and evaluate VAE + WDIWCD (best model)
**Optimal hyperparameters:** a=0.80, b=0.48, c=0.87, latent_dim=16, hidden_dim=500

![VAE-WDIWCD results](../figures/vae_wdiwcd_accuracy.png)


In [ ]:
from models.vae_wdiwcd import VAE_WDIWCD

set_seed(SEED)
model_vae_w = VAE_WDIWCD(input_dim, hidden_dim=500, latent_dim=16,
                          n_classes=10, hist_bins=32, n_heads=1)
opt = torch.optim.Adam(model_vae_w.parameters(), lr=0.001)

for epoch in tqdm(range(30), desc="VAE+WDIWCD"):  # use 100 for full reproduction
    for x, y in train_loader:
        opt.zero_grad()
        recon, mu, log_var, wloss, _, _ = model_vae_w(x, y)
        loss = VAE_WDIWCD.combined_loss(recon.float(), x, mu, log_var, wloss,
                                        a=0.80, b=0.48)
        if not torch.isnan(loss): loss.backward(); opt.step()

model_vae_w.eval()
with torch.no_grad():
    _, mu_tr, _, _, _, _ = model_vae_w(tr_x, torch.tensor(tr_y))
    _, mu_te, _, _, _, _ = model_vae_w(te_x, torch.tensor(te_y))
    recon_te, _, _, _, _, _ = model_vae_w(te_x, torch.tensor(te_y))
acc_vw = knn_accuracy(mu_tr.numpy(), tr_y, mu_te.numpy(), te_y, k=50)
from utils.metrics import reconstruction_mse
mse    = reconstruction_mse(te_x.numpy(), recon_te.numpy())
print(f"VAE+WDIWCD  →  kNN Accuracy: {acc_vw*100:.1f}%  |  MSE: {mse:.5f}")


## 4. Summary table (5-run average)
Run `python classification/train.py --config classification/configs/vae_wdiwcd_optimal.yaml`
to reproduce the full 5-run / 10-run averaged results from Tables 3 and 8 of the paper.

| Method | kNN Acc (MNIST) | kNN Acc (Gender) | MSE |
|---|---|---|---|
| Only VAE | 84.2 | 68.8 | 0.021 |
| Only WDIWCD | 88.9±1.89^ | 72.6±1.53^ | 0.016±0.004^ |
| **VAE+WDIWCD** | **89.4±2.09^** | **75.5±2.70^** | **0.019±0.002^** |
| VAE+WDDWCC | 89.6±2.18^ | 78.6±3.95^ | 0.020±0.001^ |

*^ p<0.05 vs. Only VAE baseline (paired t-test)*